# CPI trajectory visualizations

Two figures, both showing **R_ctx** (share of Layer-0-passed items whose final,
judge-backstopped label is CTX) across every saved checkpoint:

1. **Aggregate** -- all strata pooled.
2. **Per-stratum** -- capitals / biographies / world facts separately.

The final label is bug-invariant (validated: <=0.002 shift under the tokenisation
fix), so the full un-rescored sweep is safe to use for both. Faint line = raw,
bold line = 3-checkpoint moving average. Phase picks come from `checkpoints.yaml`.

In [ ]:
import json, glob, re, os
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RESULTS_DIR = {"alpaca": "../../results/cpi-results/alpaca-results",
               "tulu":   "../../results/cpi-results/tulu-results"}
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

cfg = yaml.safe_load(open("checkpoints.yaml"))
PHASES = {run: {k: int(v.split("-")[1]) for k, v in cfg["runs"][run]["phases"].items() if v}
          for run in ["alpaca", "tulu"]}
print(PHASES)

# validated categorical palette (see dataviz skill, references/palette.md)
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK_PRIMARY, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial"],
    "axes.edgecolor": BASELINE,
    "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_MUTED,
    "ytick.color": INK_MUTED,
    "text.color": INK_PRIMARY,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.grid": True,
    "axes.axisbelow": True,
})

In [ ]:
STRATA = {"country_capitals": "capitals", "famous_biographies": "biographies", "world_facts": "world facts"}
RUN_TITLE = {"alpaca": "Alpaca SFT", "tulu": "Tulu SFT"}
SMOOTH_WINDOW = 3  # alpaca has only 16 checkpoints; a wider window flattens its real step-100 peak

def all_steps(run):
    """Numeric checkpoint steps only (skips checkpoint_stepinf_metrics.json)."""
    files = glob.glob(f"{RESULTS_DIR[run]}/checkpoint_step*_metrics.json")
    return sorted(int(m.group(1)) for f in files if (m := re.search(r"step(\d+)_metrics", f)))

def rctx_trajectory(run):
    """steps, pooled R_ctx, {stratum: R_ctx list} -- final label, full sweep."""
    steps, pooled, by_stratum = [], [], {s: [] for s in STRATA}
    for step in all_steps(run):
        d = json.load(open(f"{RESULTS_DIR[run]}/checkpoint_step{step}_metrics.json", encoding="utf-8"))
        steps.append(step)
        pooled.append(d["aggregate"]["final"]["R_ctx"]["mean"])
        for s in STRATA:
            by_stratum[s].append(d["per_stratum"][s]["final"]["R_ctx"]["mean"])
    return steps, pooled, by_stratum

def smooth(values, window=SMOOTH_WINDOW):
    """Centred moving average, edge-replicated so the length is preserved."""
    values = np.asarray(values, dtype=float)
    pad = window // 2
    padded = np.pad(values, (pad, pad), mode="edge")
    return np.convolve(padded, np.ones(window) / window, mode="valid")[:len(values)]

def style_axis(ax, run):
    for name, step in PHASES[run].items():
        ax.axvline(step, color=INK_MUTED, linewidth=0.8, linestyle="--", zorder=1)
        ax.text(step, 1.05, name, transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=8, color=INK_SECONDARY)
    ax.set_title(RUN_TITLE[run], fontsize=12, color=INK_PRIMARY, loc="left", pad=22)
    ax.set_xlabel("Training step", fontsize=10)
    ax.set_ylabel("R_ctx (context reliance)", fontsize=10)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)

DATA = {run: rctx_trajectory(run) for run in ["alpaca", "tulu"]}

## Figure 1 -- aggregate R_ctx

In [ ]:
# Independent y-axes: alpaca's ~11pp swing is real but flattens on tulu's wider range.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))

for ax, run in zip(axes, ["alpaca", "tulu"]):
    steps, pooled, _ = DATA[run]
    ax.plot(steps, pooled, color=BLUE, linewidth=1.2, alpha=0.5, zorder=2)
    ax.plot(steps, smooth(pooled), color=BLUE, linewidth=2.5, zorder=3)
    lo, hi = min(pooled), max(pooled)
    pad = (hi - lo) * 0.15
    ax.set_ylim(lo - pad, hi + pad)
    style_axis(ax, run)

fig.suptitle("Context reliance over SFT", fontsize=13, color=INK_PRIMARY, x=0.08, ha="left", y=1.06)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/aggregate_trajectory.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/aggregate_trajectory.pdf", bbox_inches="tight")
plt.show()

## Figure 2 -- per-stratum R_ctx

In [ ]:
STRATUM_COLOR = {"country_capitals": BLUE, "famous_biographies": ORANGE, "world_facts": AQUA}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)

for ax, run in zip(axes, ["alpaca", "tulu"]):
    steps, _, by_stratum = DATA[run]
    for s, label in STRATA.items():
        ax.plot(steps, by_stratum[s], color=STRATUM_COLOR[s], linewidth=1.2, alpha=0.4, zorder=2)
        ax.plot(steps, smooth(by_stratum[s]), color=STRATUM_COLOR[s], linewidth=2.5, zorder=3, label=label)
    style_axis(ax, run)
axes[1].set_ylabel("")

axes[0].legend(loc="upper right", frameon=False, fontsize=9)
fig.suptitle("Context reliance over SFT, by stratum", fontsize=13, color=INK_PRIMARY, x=0.08, ha="left", y=1.06)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/stratum_trajectory.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{FIG_DIR}/stratum_trajectory.pdf", bbox_inches="tight")
plt.show()